In [0]:
%pip install squarify

In [0]:
import squarify, pandas as pd

data = df.toPandas()
data["name"] = data["channel"]
data["size"] = data["measure(units_sold)"]
data = data.drop(["channel", "measure(units_sold)"], axis=1)
rects = squarify.squarify(squarify.normalize_sizes(data["size"], 600, 400), 0, 0, 600, 400)
df = pd.DataFrame(rects).assign(name=data["name"], size=data["size"])
df["x2"], df["y2"] = df.x + df.dx, df.y + df.dy

In [0]:
sdf = spark.createDataFrame(df)

In [0]:
sdf.write.saveAsTable("encore_dev.subramanian.sales_data_treemap", mode="overwrite")

#Squarify for SQL Warehouse to Test

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

# Auto-discover the first available pro/serverless SQL warehouse
w = WorkspaceClient()
warehouse = next(
    (wh for wh in w.warehouses.list()
     if wh.warehouse_type and wh.warehouse_type.value in ("PRO", "SERVERLESS")),
    None
)
assert warehouse, "No pro or serverless SQL warehouse found. Please create one in the SQL Warehouses UI."
print(f"Using warehouse: {warehouse.name} ({warehouse.id})")

# Build the dollar-quote delimiter at runtime (avoids parser stripping)
DQ = "$" + "$"

UDF_BODY = """
def normalize_sizes(sizes, dx, dy):
    total = sum(sizes)
    if total == 0:
        return [0.0] * len(sizes)
    return [float(s) * dx * dy / total for s in sizes]

def worst_ratio(row, length):
    s = sum(row)
    thickness = s / length
    ratios = []
    for r in row:
        w = r / thickness
        ratios.append(max(thickness / w, w / thickness))
    return max(ratios)

def layout_row(row, x, y, dx, dy):
    s = sum(row)
    rects = []
    if dx >= dy:
        width = s / dy
        ry = y
        for r in row:
            h = r / width
            rects.append({"x": x, "y": ry, "dx": width, "dy": h})
            ry += h
        return rects, (x + width, y, dx - width, dy)
    else:
        height = s / dx
        rx = x
        for r in row:
            w = r / height
            rects.append({"x": rx, "y": y, "dx": w, "dy": height})
            rx += w
        return rects, (x, y + height, dx, dy - height)

def squarify(norm, x, y, dx, dy):
    rects, row = [], []
    while norm and dx > 0 and dy > 0:
        length = min(dx, dy)
        candidate = row + [norm[0]]
        if not row or worst_ratio(candidate, length) <= worst_ratio(row, length):
            row, norm = candidate, norm[1:]
        else:
            placed, (x, y, dx, dy) = layout_row(row, x, y, dx, dy)
            rects.extend(placed)
            row = []
    if row:
        placed, _ = layout_row(row, x, y, dx, dy)
        rects.extend(placed)
    return rects

if not sizes or dx <= 0 or dy <= 0:
    return []

norm = normalize_sizes([float(s) for s in sizes], float(dx), float(dy))
rects = squarify(norm, 0.0, 0.0, float(dx), float(dy))

return [
    {
        "name": labels[i] if labels and i < len(labels) else str(i),
        "x":    rect["x"],
        "y":    rect["y"],
        "dx":   rect["dx"],
        "dy":   rect["dy"],
        "size": float(sizes[i]),
        "x2":   rect["x"] + rect["dx"],
        "y2":   rect["y"] + rect["dy"],
    }
    for i, rect in enumerate(rects)
]
"""

UDF_SQL = f"""
CREATE OR REPLACE FUNCTION encore_dev.subramanian.squarify_treemap(
  sizes  ARRAY<DOUBLE>,
  labels ARRAY<STRING>,
  dx     DOUBLE,
  dy     DOUBLE
)
RETURNS ARRAY<STRUCT<name STRING, x DOUBLE, y DOUBLE, dx DOUBLE, dy DOUBLE, size DOUBLE, x2 DOUBLE, y2 DOUBLE>>
LANGUAGE PYTHON
COMMENT 'Squarify treemap layout. sizes: area values, labels: names, dx/dy: canvas size.'
AS {DQ}{UDF_BODY}{DQ}
"""

# Execute DDL directly against the SQL warehouse (bypasses Spark Connect parser)
response = w.statement_execution.execute_statement(
    statement=UDF_SQL,
    warehouse_id=warehouse.id,
    wait_timeout="30s",
)

if response.status.state == StatementState.SUCCEEDED:
    print("Function encore_dev.subramanian.squarify_treemap registered successfully.")
else:
    raise RuntimeError(f"DDL failed: {response.status.error}")

# --- Reference SQL (run from any SQL Warehouse) ---
# SELECT col.*
# FROM (
#   SELECT explode(
#     encore_dev.subramanian.squarify_treemap(
#       collect_list(CAST(`measure(units_sold)` AS DOUBLE)),
#       collect_list(channel),
#       600, 400
#     )
#   ) AS col
#   FROM (
#     SELECT measure(units_sold), channel
#     FROM encore_dev.subramanian.sales_data
#     GROUP BY ALL
#   )
# )

In [0]:
%sql
-- Call the UDF: collect channel sizes into arrays, compute treemap coords, and save
CREATE OR REPLACE TABLE encore_dev.subramanian.sales_data_treemap AS
SELECT col.*
FROM (
  SELECT explode(
    encore_dev.subramanian.squarify_treemap(
      collect_list(CAST(`measure(units_sold)` AS DOUBLE)),
      collect_list(channel),
      600, 400
    )
  ) AS col
  FROM (
    SELECT measure(units_sold), channel
    FROM encore_dev.subramanian.sales_data
    GROUP BY ALL
  )
);

SELECT * FROM encore_dev.subramanian.sales_data_treemap